# Day 10 Project: Templated Prompt Engine

## What You're Building

You run this notebook. It loads a JSON registry of named, versioned prompt templates (`summarize`, `classify`, `extract`), renders each with sample variables, and calls the local Ollama model. For each template it prints the filled prompt and the model's response. Output shows three separate template-driven LLM interactions, each constructed without a single hard-coded f-string in the calling code.

That's the deliverable.

---

**Concepts you'll compose:**

- **Lesson 1** — Parameterized template functions: structure lives in the template, data enters via arguments
- **Lesson 2** — `string.Template` with `$variable` placeholders and `substitute()` for safe filling
- **Lesson 3** — `PromptTemplate` dataclass: `name`, `template_str`, `required_vars`, `description` plus `validate()` and `render()`
- **Lesson 4** — Registry dict keyed by `name_vN`, saved/loaded as JSON, immutability enforced by `register()`
- **Lesson 5** — `render_and_call()`: fills the template first, passes the result to `ollama.chat()` as the user message

> Complete Exercises 1–5 before starting this project.

## Step 1: Imports and Setup

In [ ]:
# Step 1: Imports — everything you need is in the stdlib or already installed
import json
import logging
from dataclasses import dataclass
from pathlib import Path
from string import Template

import ollama

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(message)s",
)
logger = logging.getLogger(__name__)

MODEL = "llama3.2"
REGISTRY_PATH = Path("prompt_registry.json")

## Step 2: PromptTemplate Dataclass

In [ ]:
# Step 2: Implement the PromptTemplate dataclass from Lesson 3.
# Fields: name, template_str, required_vars, description
# Methods: validate(values) and render(values)

@dataclass
class PromptTemplate:
    """A self-documenting, validatable prompt template."""

    name: str
    template_str: str
    required_vars: list  # list of placeholder name strings
    description: str

    def validate(self, values: dict) -> None:
        # TODO: collect every name in required_vars that is missing from values
        # TODO: if any are missing, raise ValueError listing the template name
        #       and all missing variable names at once
        pass

    def render(self, values: dict) -> str:
        # TODO: call self.validate(values) first
        # TODO: build a string.Template from self.template_str
        # TODO: call .substitute(values) and return the filled string
        pass

## Step 3: Registry Functions

In [ ]:
# Step 3: Build the registry — a plain dict keyed by 'name_vN' strings.
# Implement register(), get_template(), save_registry(), and load_registry().

_registry: dict = {}  # {'summarize_v1': PromptTemplate, ...}


def register(pt: PromptTemplate) -> PromptTemplate:
    """Add a PromptTemplate to the registry.

    Key format: pt.name (expected to already include the version suffix, e.g. 'summarize_v1').
    Raises ValueError if the key already exists — versions are immutable.
    Returns pt unchanged so it can be used inline.
    """
    # TODO: check whether pt.name is already in _registry; raise ValueError if so
    # TODO: store pt in _registry under pt.name
    # TODO: return pt
    pass


def get_template(key: str) -> PromptTemplate:
    """Return the PromptTemplate for the given key. Raises KeyError if missing."""
    # TODO: return _registry[key]
    pass


def save_registry(path) -> None:
    """Serialise the registry to a JSON file."""
    # TODO: build a plain dict of dicts from _registry
    #   each entry: {'template_str': ..., 'required_vars': ..., 'description': ...}
    # TODO: write it with json.dump(indent=2)
    pass


def load_registry(path) -> None:
    """Load a JSON registry file and merge into the live _registry."""
    # TODO: open the file with json.load
    # TODO: for each key/entry, construct a PromptTemplate and call register()
    pass

## Step 4: Define and Register the Three Templates

In [ ]:
# Step 4: Define three PromptTemplate instances and register them.
# Templates to build:
#
#   summarize_v1  — fills $text and $max_sentences
#   classify_v1   — fills $text and $categories
#   extract_v1    — fills $entity_type and $text
#
# Register each with register(). Use string.Template $placeholder syntax.
# Do NOT use f-strings inside template_str.

# TODO: define and register SUMMARIZE

# TODO: define and register CLASSIFY

# TODO: define and register EXTRACT

print("Registered templates:", list(_registry.keys()))

## Step 5: Save the Registry to JSON

In [ ]:
# Step 5: Persist the registry to REGISTRY_PATH so it can be loaded
# in a fresh session (Lesson 4 pattern).

# TODO: call save_registry(REGISTRY_PATH)
# TODO: print the path and confirm the file exists
pass

## Step 6: render() and render_and_call()

In [ ]:
# Step 6: Implement the render-then-call pattern from Lesson 5.

def render(key: str, variables: dict) -> str:
    """Look up the template, substitute variables, return the filled string.

    No model call happens here — this is the pure construction step.
    Raises KeyError if key not found or a placeholder is missing.
    """
    # TODO: call get_template(key) to retrieve the PromptTemplate
    # TODO: call pt.render(variables) to get the filled string
    # TODO: log at DEBUG (template key + prompt length)
    # TODO: return the filled string
    pass


def render_and_call(
    key: str,
    variables: dict,
    model: str = MODEL,
    system_prompt: str = None,
) -> str:
    """Render a template and call the local Ollama model.

    Args:
        key:           Registry key (e.g. 'summarize_v1').
        variables:     Placeholder values to fill into the template.
        model:         Ollama model name.
        system_prompt: Optional system message prepended to the messages list.

    Returns:
        The model's reply as a plain string.
    """
    # TODO: Step 1 — render: call render(key, variables) to get the user message string
    # TODO: Step 2 — build: construct the messages list
    #   if system_prompt is given, prepend {'role': 'system', 'content': system_prompt}
    #   always append {'role': 'user', 'content': <rendered string>}
    # TODO: Step 3 — call: ollama.chat(model=model, messages=messages)
    # TODO: return response['message']['content']
    pass

## Step 7: Run Three Template-Driven LLM Interactions

In [ ]:
# Step 7: Drive three separate LLM calls through the render_and_call interface.
# For each call:
#   - print the template key
#   - print the rendered (filled) prompt
#   - print the model's response
#
# No f-strings allowed in the calling code below — all prompt text lives in the registry.

SAMPLE_TEXT = (
    "The Python programming language was created by Guido van Rossum "
    "and first released in 1991. It emphasises code readability and "
    "supports multiple programming paradigms including procedural, "
    "object-oriented, and functional programming."
)

# TODO: call 1 — summarize_v1
#   variables: text=SAMPLE_TEXT, max_sentences='2'


# TODO: call 2 — classify_v1
#   variables: text=SAMPLE_TEXT, categories='history, technology, science, art'


# TODO: call 3 — extract_v1
#   variables: entity_type='year', text=SAMPLE_TEXT

## Step 8: Confirmation

In [ ]:
# Step 8: Print the deliverable confirmation.
# TODO: print a summary of what was produced:
#   - how many templates are in the registry
#   - that the registry was saved to REGISTRY_PATH
#   - that three template-driven LLM calls completed
pass